In [14]:
from argparse import ArgumentParser
import xarray as xr
from matplotlib import pyplot as plt
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import sys

sys.path.insert(
    1, "/home/hk-project-pai00005/xo8179/neural_lam_fork/neural-lam/evaluation"
)
import plot_utils as plot
import numpy as np

save_folder = "/home/hk-project-pai00005/xo8179/neural_lam_fork/neural-lam/eval_nextgems_new/values/"


def create_folder_if_not_exists(folder):
    from pathlib import Path

    Path(folder).mkdir(parents=True, exist_ok=True)


def get_temporal_index_in_dataset(timestamp, ds):
    # Suppose your timestamp is
    target_time = timestamp
    
    # get the index object from the DataArray
    time_index = ds.get_index("time")
    
    # find the integer position
    idx = time_index.get_loc(target_time)
    return idx
    
def get_true_data_matching_forecast_timestamps(ds_truth, ds_forecast, prediction_steps):
    '''
    prediction_steps should be 0 based -> 0 means using the first prediction of the model for that timestamp
    '''
    first_timestamp_forecast = ds_forecast.time[0].values
    last_timestamp_forecast = ds_forecast.time[-1].values
    idx_truth_first = get_temporal_index_in_dataset(first_timestamp_forecast, ds_truth) + prediction_steps+1
    idx_truth_last = get_temporal_index_in_dataset(last_timestamp_forecast, ds_truth) + prediction_steps+1
    new_forecast = ds_forecast.copy().isel(prediction_timedelta=prediction_steps)
    new_truth = ds_truth.isel(time=slice(idx_truth_first, idx_truth_last + 1)).isel(time=slice(0, None, 2))
    new_forecast = new_forecast.sortby('time')
    _, index = np.unique(new_forecast['time'], return_index=True)
    new_forecast = new_forecast.isel(time=np.sort(index))
    
    for t in range(len(new_forecast.time.values)-1):
        if not new_forecast.time.values[t+1] - new_forecast.time.values[t] == 43200000000000:
            print(f'Error with {new_forecast.time.values[t]} and {new_forecast.time.values[t+1]}: difference is {str(new_forecast.time.values[t+1] - new_forecast.time.values[t])}')
    new_forecast = new_forecast.assign_coords({'time': new_truth['time']})
    return new_truth, new_forecast


def get_stacked_true_data_matching_forecast_timestamps(ds_truth, ds_forecast):
    """
    prediction_steps should be 0 based -> 0 means using the first prediction of the model for that timestamp
    """
    first_timestamp_forecast = ds_forecast.time[0].values
    last_timestamp_forecast = ds_forecast.time[-1].values
    idx_truth_first = get_temporal_index_in_dataset(
        first_timestamp_forecast, ds_truth
    )
    idx_truth_last = get_temporal_index_in_dataset(
        last_timestamp_forecast, ds_truth
    )
    new_forecast = ds_forecast.copy()
    new_truth = ds_truth.isel(
        time=slice(idx_truth_first, idx_truth_last + 1)
    ).isel(time=slice(0, None, 2))
    new_forecast = new_forecast.sortby("time")
    _, index = np.unique(new_forecast["time"], return_index=True)
    new_forecast = new_forecast.isel(time=np.sort(index))
    new_forecast = new_forecast.chunk({"latitude": -1, "longitude": -1,"time": 50})
    new_truth = new_truth.chunk({"latitude": -1, "longitude": -1,"time": 50})
    new_truth = new_truth.assign_coords({'prediction_timedelta': new_forecast['prediction_timedelta']})
    return new_truth, new_forecast


def build_truth_with_leadtime(truth_ds, max_lead_steps=40):
    """
    truth_ds: Dataset with dimension (time, lat, lon, level)
    Creates a dataset with dimension (time, prediction_timedelta, lat, lon, level)

    lead step k uses truth_ds shifted by k steps forward in time.
    """
    truth_ds = truth_ds.chunk({"time": 50})
    lead_times = np.arange(1, max_lead_steps + 1)

    # Build stacked truth array
    truth_list = []
    for k in lead_times:
        truth_shifted = truth_ds.shift(
            time=-k
        )  # shift backwards so truth[t,k] = truth[t+k]
        truth_list.append(truth_shifted)

    # Combine along new dimension
    truth_with_lead = xr.concat(truth_list, dim="prediction_timedelta")

    # Assign actual timedelta
    truth_with_lead = truth_with_lead.assign_coords(
        prediction_timedelta=xr.DataArray(
            (lead_times * np.timedelta64(6, "h")).astype("timedelta64[ns]"),
            dims=("prediction_timedelta",),
        )
    )

    # Drop times near the end that became NaN
    truth_with_lead = truth_with_lead.dropna(dim="time", how="any")

    return truth_with_lead

In [15]:
model_idx=1
past_int=0
var_start=0
    
past = False if past_int == 0 else True

model_resolution = [
    "z64_era5",
    "z64",
    "z128",
    "z256",
    "z64 hierarchical",
    "persistence",
]

current_model_resolution = model_resolution[model_idx]
print(
    f'Calculating RMSE for model {current_model_resolution} in {"past" if past else "future"}'
)

selected_model_past = plot.PAST_MODELS_NG_era5[model_idx]
selected_model_future = plot.FUTURE_MODELS_NG_era5[model_idx]
print(f'Evaluating model: {selected_model_past["model_name"]}')

base_save_folder_past = selected_model_past["plot_folder"]
base_save_folder_future = selected_model_future["plot_folder"]

ng_past_forecasts = xr.open_zarr(
    selected_model_past["forecasts"], decode_timedelta=True
).astype(np.float32)
ng_future_forecasts = xr.open_zarr(
    selected_model_future["forecasts"], decode_timedelta=True
).astype(np.float32)

ng_past_truth = xr.open_zarr(selected_model_past["ground_truth"]).astype(
    np.float32
)
ng_future_truth = xr.open_zarr(
    selected_model_future["ground_truth"]
).astype(np.float32)

create_folder_if_not_exists(base_save_folder_past)
create_folder_if_not_exists(base_save_folder_future)

print("Stacking truth...")
truth_stacked = build_truth_with_leadtime(
    ng_past_truth if past else ng_future_truth
)


print("Selecting matching truth / forecasts...")
truth_ds, forecast_ds = get_stacked_true_data_matching_forecast_timestamps(
    truth_stacked, ng_past_forecasts if past else ng_future_forecasts
)


print("Preparing weights...")
lat = truth_ds["latitude"]
weights = np.cos(np.deg2rad(lat))
weights = xr.DataArray(weights, coords={"latitude": lat}, dims=["latitude"])
weights = weights / weights.sum()

all_rmse = {}
rmse_maps = {}

Calculating RMSE for model z64 in future
Evaluating model: NextGEMS z64 Multilevel Graph
Stacking truth...
Selecting matching truth / forecasts...
Preparing weights...


In [22]:
for key in range(1, len(plot.KEY_VARIABLES)):

    display_name, unit, variable, plevel = plot.KEY_VARIABLES[key]

    print(f"Current var (RMSE): {display_name}")

    # Select variable
    truth = truth_ds[variable]
    forecast = forecast_ds[variable]

    # Level selection
    if plevel is not None:
        truth = truth.sel(level=plevel)
        forecast = forecast.sel(level=plevel)
    

    # Broadcast weights
    W2d = weights.broadcast_like(truth.isel(time=0, prediction_timedelta=0))
    W = W2d.broadcast_like(truth)

    # Weighted MSE
    mse = ((forecast - truth) ** 2 * W).sum(
        dim=("time", "latitude", "longitude")
    ) / W2d.sum(dim=("latitude", "longitude"))

    rmse = np.sqrt(mse)

    mse_map = ((forecast - truth) ** 2).mean(dim="time")
    rmse_map = np.sqrt(mse_map)  # (prediction_timedelta, lat, lon)

    if plevel:
        rmse_maps[variable] = rmse_map.drop_vars("level")
    else:
        rmse_maps[variable] = rmse_map

    print(f"RMSE map for {variable}: {rmse_maps[variable].shape}")

    # Convert to numpy
    rmse_values = rmse.values  # shape (40,)
    all_rmse[variable] = rmse_values

    print(all_rmse[variable])
    # Save
    import pandas as pd

    df_rmse = pd.DataFrame(all_rmse).T
    print(all_rmse)
    print(df_rmse)
    df_rmse.columns = [f"step_{i+1}" for i in range(40)]

    save_path = (
        save_folder+
        f"rmse_all_variables_{current_model_resolution}_{'past' if past else 'future'}{var_start if var_start > 0 else ''}.csv"
    )

    df_rmse.to_csv(save_path)
    print("Saved:", save_path)

    ds_rmse_maps = xr.Dataset({var: rmse_maps[var] for var in rmse_maps})

    mode = "w"

    map_save = (
        save_folder+
        f"rmse_maps_{current_model_resolution}_{'past' if past else 'future'}{var_start if var_start > 0 else ''}.zarr"
    )
    ds_rmse_maps['prediction_timedelta'].attrs.pop("units", None)
    ds_rmse_maps.to_zarr(map_save, mode=mode)
    print("Saved RMSE maps:", map_save)

Current var (RMSE): Temperature at 850hPa
RMSE map for temperature: (40, 128, 64)
[ 33.51467971  45.08077345  51.13673448  55.46422619  60.21089698
  64.49301238  68.32559895  72.13227681  76.3190244   80.64087303
  84.93707461  89.40281854  94.09190091  98.94485698 103.75559473
 108.70817395 113.73116361 118.83519318 123.80884588 128.84620336
 133.85442809 138.87169386 143.71920882 148.57434825 153.34765469
 158.08912879 162.59628184 167.05487871 171.3904803  175.6257998
 179.61709305 183.52309466 187.30199567 190.97525768 194.40402105
 197.74692516 200.96725406 204.09804521 206.9973472  209.79274783]
{'geopotential': array([ 3019.77758561,  4305.55001102,  5643.30702137,  6884.51534024,
        8182.4600301 ,  9429.17904603, 10682.26634107, 11950.66242609,
       13222.90286519, 14512.29893033, 15787.18818634, 17093.02032132,
       18374.98097784, 19692.46187688, 20975.59579933, 22296.82876298,
       23580.45602482, 24894.51588714, 26160.20680709, 27454.75690661,
       28700.17296

In [21]:
print(ds_rmse_maps)
ds_rmse_maps['prediction_timedelta'].attrs.pop("units", None)
ds_rmse_maps.to_zarr(map_save, mode=mode)

<xarray.Dataset> Size: 1MB
Dimensions:               (latitude: 64, longitude: 128,
                           prediction_timedelta: 40)
Coordinates:
  * latitude              (latitude) float64 512B -90.0 -87.14 ... 87.14 90.0
  * longitude             (longitude) float64 1kB 0.0 2.812 ... 354.4 357.2
  * prediction_timedelta  (prediction_timedelta) timedelta64[ns] 320B 00:00:0...
Data variables:
    geopotential          (prediction_timedelta, longitude, latitude) float32 1MB dask.array<chunksize=(1, 128, 64), meta=np.ndarray>


In [8]:
key=0

display_name, unit, variable, plevel = plot.KEY_VARIABLES[key]

print(f"Current var (RMSE): {display_name}")

# Select variable
truth = truth_ds[variable]
forecast = forecast_ds[variable]
truth = truth.assign_coords({'prediction_timedelta': forecast['prediction_timedelta']})

# Level selection
if plevel is not None:
    truth = truth.sel(level=plevel)
    forecast = forecast.sel(level=plevel)

print(forecast-truth)

Current var (RMSE): Geopotential at 500hPa
<xarray.DataArray 'geopotential' (time: 2900, prediction_timedelta: 40,
                                  longitude: 128, latitude: 64)> Size: 4GB
dask.array<sub, shape=(2900, 40, 128, 64), dtype=float32, chunksize=(1, 1, 128, 64), chunktype=numpy.ndarray>
Coordinates:
  * latitude              (latitude) float64 512B -90.0 -87.14 ... 87.14 90.0
    level                 int64 8B 500
  * longitude             (longitude) float64 1kB 0.0 2.812 ... 354.4 357.2
  * prediction_timedelta  (prediction_timedelta) timedelta64[ns] 320B 00:00:0...
  * time                  (time) datetime64[ns] 23kB 2046-01-01T06:00:00 ... ...


In [9]:
# Broadcast weights
W2d = weights.broadcast_like(truth.isel(time=0, prediction_timedelta=0))
W = W2d.broadcast_like(truth)

# Weighted MSE
mse = ((forecast - truth) ** 2 * W).sum(
    dim=("time", "latitude", "longitude")
) / W2d.sum(dim=("latitude", "longitude"))

rmse = np.sqrt(mse)
print(rmse)

<xarray.DataArray (prediction_timedelta: 40)> Size: 320B
dask.array<sqrt, shape=(40,), dtype=float64, chunksize=(1,), chunktype=numpy.ndarray>
Coordinates:
    level                 int64 8B 500
  * prediction_timedelta  (prediction_timedelta) timedelta64[ns] 320B 00:00:0...


In [ ]:
mse_map = ((forecast - truth) ** 2).mean(dim="time")
rmse_map = np.sqrt(mse_map)  # (prediction_timedelta, lat, lon)

if plevel:
    rmse_maps[variable] = rmse_map.drop_vars("level")
else:
    rmse_maps[variable] = rmse_map

print(f"RMSE map for {variable}: {rmse_maps[variable].shape}")

# Convert to numpy
rmse_values = rmse.values  # shape (40,)
all_rmse[variable] = rmse_values

print(all_rmse[variable])
# Save
import pandas as pd

df_rmse = pd.DataFrame(all_rmse).T
print(all_rmse)
print(df_rmse)
df_rmse.columns = [f"step_{i+1}" for i in range(40)]

save_path = (
    save_folder+
    f"rmse_all_variables_{current_model_resolution}_{'past' if past else 'future'}{var_start if var_start > 0 else ''}.csv"
)

df_rmse.to_csv(save_path)
print("Saved:", save_path)

ds_rmse_maps = xr.Dataset({var: rmse_maps[var] for var in rmse_maps})

mode = "w"

map_save = (
    save_folder+
    f"rmse_maps_{current_model_resolution}_{'past' if past else 'future'}{var_start if var_start > 0 else ''}.zarr"
)
ds_rmse_maps.to_zarr(map_save, mode=mode)
print("Saved RMSE maps:", map_save)

Current var (RMSE): Geopotential at 500hPa
RMSE map for geopotential: (0, 128, 64)
[]
{'geopotential': array([], dtype=float64)}
Empty DataFrame
Columns: []
Index: [geopotential]


ValueError: Length mismatch: Expected axis has 0 elements, new values have 40 elements